# Logistic Regression

**INDE 577 / CMOR 438 — Rice University**  
**Instructor:** Randy R. Davila, PhD

---

## Overview

Logistic Regression is a **supervised learning** algorithm for **binary (and multi-class) classification**. Despite the name, it is a classification algorithm that models the probability that an instance belongs to a particular class.

## Mathematical Background

### Sigmoid Function

The sigmoid (logistic) function maps any real value to the range $(0, 1)$:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

### Model

The predicted probability is:

$$\hat{p} = \sigma(\mathbf{w}^T \mathbf{x} + b) = \frac{1}{1 + e^{-(\mathbf{w}^T \mathbf{x} + b)}}$$

The predicted class label is:

$$\hat{y} = \begin{cases} 1 & \text{if } \hat{p} \geq 0.5 \\ 0 & \text{otherwise} \end{cases}$$

### Loss Function: Binary Cross-Entropy

$$\mathcal{L}(\mathbf{w}, b) = -\frac{1}{n} \sum_{i=1}^{n} \left[ y_i \log(\hat{p}_i) + (1 - y_i) \log(1 - \hat{p}_i) \right] + \frac{\lambda}{2} \|\mathbf{w}\|^2$$

The second term is **L2 regularization** to prevent overfitting.

### Gradient Descent Update

$$\mathbf{w} \leftarrow \mathbf{w} - \alpha \left( \frac{1}{n} \mathbf{X}^T (\hat{\mathbf{p}} - \mathbf{y}) + \lambda \mathbf{w} \right)$$

$$b \leftarrow b - \alpha \cdot \frac{1}{n} \sum_{i=1}^{n}(\hat{p}_i - y_i)$$

where $\alpha$ is the learning rate.

### Decision Boundary

The decision boundary is the hyperplane where $\hat{p} = 0.5$, i.e., where $\mathbf{w}^T \mathbf{x} + b = 0$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.preprocessing import StandardScaler as SklearnScaler
from sklearn.linear_model import LogisticRegression as SklearnLR
from sklearn.model_selection import train_test_split as sklearn_split
from sklearn.metrics import classification_report as sklearn_report
import warnings
warnings.filterwarnings('ignore')

from rice_ml import LogisticRegression
from rice_ml.processing.preprocessing import StandardScaler, train_test_split
from rice_ml.processing.metrics import accuracy_score, confusion_matrix, classification_report

print("Libraries loaded successfully!")
np.random.seed(42)

## 1. Toy Dataset: Visualizing the Decision Boundary

We start with a simple 2D dataset so we can visualize the decision boundary.

In [ ]:
# Generate 2D binary classification data
X_toy, y_toy = make_classification(
    n_samples=300, n_features=2, n_redundant=0,
    n_informative=2, random_state=42, n_clusters_per_class=1
)

# Scale features
scaler = StandardScaler()
X_toy_scaled = scaler.fit_transform(X_toy)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X_toy_scaled, y_toy, test_size=0.2, random_state=42)

# Train logistic regression
model = LogisticRegression(learning_rate=0.1, n_iterations=1000, lambda_reg=0.01)
model.fit(X_train, y_train)

# Evaluate
acc = accuracy_score(y_test, model.predict(X_test))
print(f"Test Accuracy: {acc:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Plot 1: Decision boundary ---
ax = axes[0]
x_min, x_max = X_toy_scaled[:, 0].min() - 0.5, X_toy_scaled[:, 0].max() + 0.5
y_min, y_max = X_toy_scaled[:, 1].min() - 0.5, X_toy_scaled[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
ax.scatter(X_toy_scaled[y_toy == 0, 0], X_toy_scaled[y_toy == 0, 1],
           c='blue', alpha=0.6, edgecolors='k', linewidths=0.5, label='Class 0', s=30)
ax.scatter(X_toy_scaled[y_toy == 1, 0], X_toy_scaled[y_toy == 1, 1],
           c='red', alpha=0.6, edgecolors='k', linewidths=0.5, label='Class 1', s=30)

# Draw the decision boundary line: w[0]*x + w[1]*y + b = 0 => y = -(w[0]*x + b) / w[1]
w = model.weights_
b = model.bias_
x_line = np.linspace(x_min, x_max, 100)
y_line = -(w[0] * x_line + b) / w[1]
ax.plot(x_line, y_line, 'k--', linewidth=2, label='Decision boundary')
ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)
ax.set_xlabel('Feature 1', fontsize=12)
ax.set_ylabel('Feature 2', fontsize=12)
ax.set_title('Logistic Regression Decision Boundary', fontsize=13, fontweight='bold')
ax.legend()

# --- Plot 2: Training loss curve ---
ax2 = axes[1]
ax2.plot(model.loss_history_, color='crimson', linewidth=2)
ax2.set_xlabel('Iteration', fontsize=12)
ax2.set_ylabel('Binary Cross-Entropy Loss', fontsize=12)
ax2.set_title('Training Loss Curve', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.set_facecolor('#f8f8f8')

plt.tight_layout()
plt.savefig('logistic_regression_toy.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"Final loss: {model.loss_history_[-1]:.4f}")

## 2. Real Dataset: Breast Cancer Classification

The [Breast Cancer Wisconsin dataset](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html) contains 569 samples with 30 features. The task is to classify tumors as **malignant (0)** or **benign (1)**.

In [ ]:
# Load dataset
data = load_breast_cancer()
X, y = data.data, data.target
feature_names = data.feature_names
target_names = data.target_names

print(f"Dataset: {data['DESCR'].splitlines()[0]}")
print(f"Samples: {X.shape[0]}, Features: {X.shape[1]}")
print(f"Classes: {target_names} | Counts: {np.bincount(y)}")

In [ ]:
# Preprocess
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Train our model
lr_model = LogisticRegression(learning_rate=0.05, n_iterations=2000, lambda_reg=0.001)
lr_model.fit(X_train, y_train)

y_pred = lr_model.predict(X_test)
y_proba = lr_model.predict_proba(X_test)

print("=== rice_ml Logistic Regression ===")
print(classification_report(y_test, y_pred, target_names=list(target_names)))

In [ ]:
# Compare with sklearn
sk_lr = SklearnLR(max_iter=2000, C=1000, random_state=42)
sk_lr.fit(X_train, y_train)
sk_pred = sk_lr.predict(X_test)

our_acc = accuracy_score(y_test, y_pred)
sk_acc = accuracy_score(y_test, sk_pred)

print(f"rice_ml accuracy:  {our_acc:.4f}")
print(f"sklearn accuracy:  {sk_acc:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Confusion matrix ---
cm = confusion_matrix(y_test, y_pred)
ax = axes[0]
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(target_names, fontsize=11)
ax.set_yticklabels(target_names, fontsize=11)
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                fontsize=16, fontweight='bold',
                color='white' if cm[i, j] > cm.max() / 2 else 'black')
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title('Confusion Matrix', fontsize=13, fontweight='bold')
plt.colorbar(im, ax=ax)

# --- Probability distribution ---
ax2 = axes[1]
ax2.hist(y_proba[y_test == 0], bins=25, alpha=0.7, color='blue', label='Malignant (0)', edgecolor='k', linewidth=0.5)
ax2.hist(y_proba[y_test == 1], bins=25, alpha=0.7, color='red', label='Benign (1)', edgecolor='k', linewidth=0.5)
ax2.axvline(0.5, color='black', linestyle='--', linewidth=2, label='Decision threshold')
ax2.set_xlabel('Predicted Probability P(class=1)', fontsize=12)
ax2.set_ylabel('Count', fontsize=12)
ax2.set_title('Predicted Probability Distribution', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

# --- Feature importance (top 10 by |weight|) ---
ax3 = axes[2]
weights = lr_model.weights_
top_idx = np.argsort(np.abs(weights))[-10:][::-1]
colors = ['#e74c3c' if w > 0 else '#3498db' for w in weights[top_idx]]
ax3.barh(range(10), weights[top_idx], color=colors, edgecolor='k', linewidth=0.5)
ax3.set_yticks(range(10))
ax3.set_yticklabels([feature_names[i][:20] for i in top_idx], fontsize=9)
ax3.axvline(0, color='black', linewidth=0.8)
ax3.set_xlabel('Weight Magnitude', fontsize=12)
ax3.set_title('Top 10 Feature Weights', fontsize=13, fontweight='bold')
ax3.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('logistic_regression_cancer.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. Effect of Regularization Strength

L2 regularization ($\lambda$) penalizes large weights and prevents overfitting. As $\lambda$ increases, the model becomes simpler but may underfit.

In [ ]:
lambdas = [0, 0.0001, 0.001, 0.01, 0.1, 1.0, 10.0]
train_accs, test_accs = [], []

for lam in lambdas:
    m = LogisticRegression(learning_rate=0.05, n_iterations=2000, lambda_reg=lam)
    m.fit(X_train, y_train)
    train_accs.append(accuracy_score(y_train, m.predict(X_train)))
    test_accs.append(accuracy_score(y_test, m.predict(X_test)))

plt.figure(figsize=(9, 5))
plt.semilogx([l if l > 0 else 1e-5 for l in lambdas], train_accs, 'bo-', linewidth=2, markersize=7, label='Train Accuracy')
plt.semilogx([l if l > 0 else 1e-5 for l in lambdas], test_accs, 'rs--', linewidth=2, markersize=7, label='Test Accuracy')
plt.xlabel('Regularization Strength (λ)', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Effect of L2 Regularization on Accuracy', fontsize=13, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('logistic_regression_regularization.png', dpi=120, bbox_inches='tight')
plt.show()

for lam, tr, te in zip(lambdas, train_accs, test_accs):
    print(f"λ={lam:>8.4f}  train={tr:.4f}  test={te:.4f}")

## 4. Learning Rate Analysis

In [ ]:
lrs = [0.001, 0.01, 0.05, 0.1, 0.5]
colors_lr = plt.cm.plasma(np.linspace(0.1, 0.9, len(lrs)))

plt.figure(figsize=(10, 5))
for lr_val, color in zip(lrs, colors_lr):
    m = LogisticRegression(learning_rate=lr_val, n_iterations=500, lambda_reg=0.001)
    m.fit(X_train, y_train)
    plt.plot(m.loss_history_, color=color, linewidth=2, label=f'lr={lr_val}')

plt.xlabel('Iteration', fontsize=12)
plt.ylabel('Binary Cross-Entropy Loss', fontsize=12)
plt.title('Convergence for Different Learning Rates', fontsize=13, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('logistic_regression_lr.png', dpi=120, bbox_inches='tight')
plt.show()

## Summary

| Property | Value |
|---|---|
| Task | Binary classification |
| Output | Probability in $(0, 1)$ via sigmoid |
| Loss | Binary cross-entropy |
| Optimization | Gradient descent |
| Regularization | L2 (ridge) |
| Dataset | Breast Cancer Wisconsin (569 × 30) |
| rice_ml accuracy | ~97% |

**Key Takeaways:**
- Logistic regression works best when data is **linearly separable** in feature space
- L2 regularization prevents overfitting; too large $\lambda$ causes underfitting
- Feature standardization is important for gradient descent to converge efficiently
- The predicted probability $\hat{p}$ can be thresholded at values other than 0.5 to trade off precision vs recall